# Step 1 &mdash; ROI Extraction & Standardization

**Goal:** isolate the rectangular wear-track sample inside each inspection photo, rotate it to horizontal, and crop it onto a fixed canvas.

Raw photos come in different orientations, lighting conditions, and background colors (blue studio mat, dark surface, etc.). A robust ROI step is the foundation of everything downstream.

## Approach: 3-stage detection cascade

| Stage | Method | Used when |
|-------|--------|-----------|
| 1 | HSV blue-background segmentation | studio shots on the blue mat |
| 2 | Otsu thresholding on grayscale  | high-contrast scenes |
| 3 | Canny edges + morphological close | low-contrast fallback |

If any stage produces a contour whose area is &lt; 5% of the image, we drop it and fall through to the next stage.

## Detection functions

In [ ]:
import cv2
import numpy as np
from pathlib import Path


def detect_blue_background(img_hsv):
    """Segment the foreground by masking out the blue studio background."""
    mask_bg = cv2.inRange(img_hsv,
                          np.array([85, 40, 40]),
                          np.array([145, 255, 255]))
    h, w = img_hsv.shape[:2]
    if cv2.countNonZero(mask_bg) < 0.2 * h * w:
        return None  # not enough blue -> not a studio shot
    mask_fg = cv2.bitwise_not(mask_bg)
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (25, 25))
    mask_fg = cv2.morphologyEx(mask_fg, cv2.MORPH_CLOSE, kernel, iterations=2)
    mask_fg = cv2.morphologyEx(mask_fg, cv2.MORPH_OPEN,  kernel, iterations=1)
    contours, _ = cv2.findContours(mask_fg, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    return max(contours, key=cv2.contourArea) if contours else None


def detect_high_contrast(img_gray):
    """Otsu thresholding for high-contrast (bright sample on dark background)."""
    blur = cv2.GaussianBlur(img_gray, (9, 9), 0)
    _, thresh = cv2.threshold(blur, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (15, 15))
    thresh = cv2.morphologyEx(thresh, cv2.MORPH_CLOSE, kernel, iterations=3)
    contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    return max(contours, key=cv2.contourArea) if contours else None


def detect_edges(img_gray):
    """Canny edge fallback for low-contrast scenes."""
    blur = cv2.GaussianBlur(img_gray, (5, 5), 0)
    edges = cv2.Canny(blur, 50, 150)
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (15, 15))
    closed = cv2.morphologyEx(edges, cv2.MORPH_CLOSE, kernel, iterations=3)
    contours, _ = cv2.findContours(closed, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    return max(contours, key=cv2.contourArea) if contours else None

## Standardization: detect &rarr; rotate &rarr; crop &rarr; resize

After finding the contour we use `cv2.minAreaRect` to get a rotation-aware bounding box, rotate the image so the sample lies horizontally, crop with a small inward margin (to discard sticker / table edges), and finally resize to a fixed target canvas.

Two canvases are produced depending on the downstream model:

| Variant | Canvas | Target model |
|---------|--------|--------------|
| CNN  | 2000 &times; 700  | ResNet50V2 / EfficientNetV2-S |
| Swin | 3000 &times; 1050 | SwinV2-Tiny |

In [ ]:
MIN_AREA_RATIO = 0.05  # reject tiny contours
MARGIN = 15            # px inward margin


def extract_roi(img, target_w, target_h):
    """Run the cascade, rotate-align, crop, then resize to (target_w, target_h)."""
    H, W = img.shape[:2]
    hsv  = cv2.cvtColor(cv2.GaussianBlur(img, (5, 5), 0), cv2.COLOR_BGR2HSV)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    # 3-stage cascade
    for detector, arg in [(detect_blue_background, hsv),
                          (detect_high_contrast,   gray),
                          (detect_edges,           gray)]:
        c = detector(arg)
        if c is not None and cv2.contourArea(c) / (W * H) >= MIN_AREA_RATIO:
            break
    else:
        return None  # nothing usable found

    # Rotation alignment
    (cx, cy), (w_r, h_r), angle = cv2.minAreaRect(c)
    if w_r < h_r:
        angle += 90
        w_r, h_r = h_r, w_r
    M = cv2.getRotationMatrix2D((cx, cy), angle, 1.0)
    rotated = cv2.warpAffine(img, M, (W, H),
                             flags=cv2.INTER_CUBIC,
                             borderMode=cv2.BORDER_REPLICATE)

    # Crop with margin
    x0 = max(0, int(cx - w_r / 2) + MARGIN)
    y0 = max(0, int(cy - h_r / 2) + MARGIN)
    x1 = min(W, int(cx + w_r / 2) - MARGIN)
    y1 = min(H, int(cy + h_r / 2) - MARGIN)
    roi = rotated[y0:y1, x0:x1]
    if roi.size == 0:
        return None

    # Resize to fixed canvas
    return cv2.resize(roi, (target_w, target_h), interpolation=cv2.INTER_CUBIC)

## Batch process a class-folder tree

Walk every subfolder (each subfolder name is a class label), extract the ROI for each image, and save the standardized output under the same folder structure.

In [ ]:
IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}


def standardize_folder(root_in, root_out, target_w, target_h):
    root_in, root_out = Path(root_in), Path(root_out)
    root_out.mkdir(parents=True, exist_ok=True)

    n, ok = 0, 0
    for p in sorted(root_in.rglob("*")):
        if p.suffix.lower() not in IMG_EXTS:
            continue
        n += 1
        out_dir = root_out / p.relative_to(root_in).parent
        out_dir.mkdir(parents=True, exist_ok=True)

        img = cv2.imread(str(p))
        if img is None:
            continue
        roi = extract_roi(img, target_w, target_h)
        if roi is not None:
            cv2.imwrite(str(out_dir / f"{p.stem}_STD{p.suffix}"), roi)
            ok += 1

    print(f"Done. {ok} / {n} images standardized -> {root_out}")


# Example (Swin variant)
# standardize_folder("./data/raw_label_folders", "./outputs/swin/01_ROI", 3000, 1050)

## Result

On the project dataset this step recovered **1,281 / 1,281** images successfully (100% detection rate) thanks to the cascade fallback. Output is a directory of horizontally-aligned, fixed-canvas crops ready for Step 2.